# SAE Eval — cross-model comparison (SigLIP vs CLIP vs ViT)
Pulls finished runs from wandb, writes per-model CSVs here, health-checks, and
plots EV / L0 / MSE / dead-features across layers for all models on one axis.
Runs locally (no GPU). Run after the training notebooks finish.


## 1 — Install + login


In [ ]:
!pip install -q wandb pandas matplotlib
import wandb
wandb.login()


## 2 — Config: which run families to pull


In [ ]:
ENTITY  = None
PROJECT = 'siglip-sae'

# label -> TAG substring used in the run names
RUNS = {
    'siglip_topk_128': 'topk_128',      # existing SigLIP run
    'clip_topk_128':   'clip_topk_128',
    'vit_topk_128':    'vit_topk_128',
}


## 3 — Pull final metrics into one CSV per model


In [ ]:
import os, re
import wandb, pandas as pd

api = wandb.Api()
path = f'{ENTITY+"/" if ENTITY else ""}{PROJECT}'
all_runs = list(api.runs(path))

def pull(tag, label):
    rows = []
    for run in all_runs:
        if tag not in run.name:
            continue
        m = re.search(r'layer(\d+)', run.name)
        if not m:
            continue
        s = run.summary
        rows.append({
            'model': label,
            'layer': int(m.group(1)),
            'l0': s.get('metrics/l0'),
            'explained_variance': s.get('metrics/explained_variance'),
            'mse_loss': s.get('losses/mse_loss'),
            'dead_features': s.get('sparsity/dead_features'),
            'state': run.state,
        })
    df = pd.DataFrame(rows).drop_duplicates('layer').sort_values('layer').reset_index(drop=True)
    out = f'{label}.csv'
    df.to_csv(out, index=False)
    print(f'{label}: {len(df)} layers -> {out}')
    return df

frames = {label: pull(tag, label) for label, tag in RUNS.items()}


## 4 — Health check across models


In [ ]:
for label, df in frames.items():
    issues = []
    for _, r in df.iterrows():
        lyr = int(r['layer'])
        if r['state'] != 'finished':
            issues.append(f'{label} layer {lyr}: state={r["state"]}')
        if pd.notna(r['l0']) and abs(r['l0'] - 128) > 4:
            issues.append(f'{label} layer {lyr}: L0={r["l0"]:.0f} (expected 128)')
    missing = sorted(set(range(12)) - set(df['layer'].astype(int)))
    if missing:
        issues.append(f'{label}: missing layers {missing}')
    print(f'--- {label} ---')
    print('\n'.join(issues) if issues else '  all 12 healthy (finished, L0~128)')


## 5 — Plot all models on one axis


In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt

frames_list = []
for f in sorted(glob.glob('*.csv')):
    d = pd.read_csv(f)
    if 'model' not in d.columns:
        continue
    frames_list.append(d)
alldf = pd.concat(frames_list, ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
panels = [('explained_variance','Explained variance (higher=better)',axes[0,0]),
          ('l0','L0 (active features / patch)',axes[0,1]),
          ('mse_loss','MSE (raw; not cross-layer comparable)',axes[1,0]),
          ('dead_features','Dead features',axes[1,1])]
for col, title, ax in panels:
    if col not in alldf.columns:
        ax.set_visible(False); continue
    for model, g in alldf.dropna(subset=[col]).groupby('model'):
        g = g.sort_values('layer')
        ax.plot(g['layer'], g[col], marker='o', label=model)
    ax.set_title(title); ax.set_xlabel('layer'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('comparison_ev_by_model.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved comparison_ev_by_model.png')
